# 1f. Broadcasting

About 20 minutes.

One rule, and one way it hurts you.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False, linewidth=110)


class _Todo:
    def __repr__(self):
        return "<fill me in>"


TODO = _Todo()


def S(t):
    """Shape as a tuple. Type this constantly."""
    return tuple(t.shape)


def check(label, got, want=None, shape=None, dtype=None, atol=1e-5, rtol=1e-4):
    """Check one answer. Prints what went wrong, then raises."""
    bad = []
    if isinstance(got, _Todo):
        bad.append("not filled in yet")
    else:
        if shape is not None:
            if not torch.is_tensor(got):
                bad.append(f"expected a tensor, got {type(got).__name__}")
            elif S(got) != tuple(shape):
                bad.append(f"shape is {S(got)}, want {tuple(shape)}")
        if dtype is not None and torch.is_tensor(got) and got.dtype != dtype:
            bad.append(f"dtype is {got.dtype}, want {dtype}")
        if want is not None:
            g = got if torch.is_tensor(got) else torch.as_tensor(got)
            w = want if torch.is_tensor(want) else torch.as_tensor(want)
            if S(g) != S(w):
                bad.append(f"shape is {S(g)}, want {S(w)}")
            elif not torch.allclose(g.double(), w.double(), atol=atol, rtol=rtol):
                bad.append(f"values differ, largest gap "
                           f"{(g.double() - w.double()).abs().max().item():.3e}")
    if bad:
        print(f"FAIL  {label}")
        for b in bad:
            print(f"      {b}")
        raise AssertionError(label)
    print(f"ok    {label}")


def same(label, a, b):
    """Two shapes or tuples must match."""
    if isinstance(a, _Todo):
        print(f"FAIL  {label}: not filled in yet")
        raise AssertionError(label)
    if tuple(a) != tuple(b):
        print(f"FAIL  {label}\n      you said {tuple(a)}, it is {tuple(b)}")
        raise AssertionError(label)
    print(f"ok    {label} = {tuple(a)}")

## The rule

PyTorch right-aligns the two shapes, then compares axis by axis from the
right:

- equal lengths: fine
- one of them is 1: that side is repeated
- a missing axis: treated as 1
- anything else: error

In [ ]:
for a, b in [((1, 4, 1, 2), (1, 1, 2)), ((4, 1), (1, 3)), ((2, 3), (3,))]:
    print(f"{str(a):>14} + {str(b):>10} -> {S(torch.zeros(a) + torch.zeros(b))}")

In [ ]:
try:
    torch.zeros(4, 3) + torch.zeros(4)
except RuntimeError as e:
    print("(4,3) + (4,) ->", str(e).split("at non-singleton")[0].strip())

`(4,)` right-aligns onto the **last** axis, which is 3. Not the first.

## The silent case

"A missing axis is treated as 1" is what makes this dangerous. On a square
tensor there is no error and no warning.

In [ ]:
ones = torch.ones(4, 4)
col = torch.tensor([1.0, 2.0, 3.0, 4.0])

print("(4,4) * (4,)        col 0 =", (ones * col)[:, 0].tolist(), " row-scaled")
print("(4,4) * (4,)[:,None] col 0 =", (ones * col[:, None])[:, 0].tolist(), " col-scaled")

Both are `(4, 4)`. One is what you meant.

The defence is to write the blank axis explicitly rather than relying on
alignment. Rectangular shapes help too: `4096`, `14336`, `32000` mean you
usually get an exception instead of a wrong answer.

## Drill 1 (easy)

`w` is an `(I, D)` weight matrix. `col_scale` has one value per **input
column**; `row_scale` has one per **output row**. Apply each.

In [ ]:
I, D = 16, 8
w = torch.ones(I, D)
col_scale = torch.arange(1, D + 1).float()
row_scale = torch.arange(1, I + 1).float()

col_scaled = TODO
row_scaled = TODO

check("col_scaled row 0", col_scaled[0], want=col_scale)
check("row_scaled col 0", row_scaled[:, 0], want=row_scale)

The packed weight format you meet in **1j** carries a per-input-column scale
exactly like `col_scale`.

## Drill 2 (medium)

BioMistral attends over a sliding window of 4096. A query at position `p` may
see key position `j` when

```text
    j <= p          no looking ahead, itself included
and j >  p - W      nothing older than W-1 steps back
```

Build `allowed` of shape `(Tq, Skv)` from `q_pos` and `kv_pos`, using blank
axes and comparisons. No loops. Watch the inclusive and exclusive ends.

In [ ]:
W = 4
q_pos = torch.tensor([4, 5, 6])
kv_pos = torch.arange(8)

allowed = TODO

check("shape", allowed, shape=(3, 8))
check("dtype is bool", allowed.dtype == torch.bool, want=True)
check("values", allowed.long(), want=torch.tensor([
    [0, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 1, 1, 1, 1, 0, 0],
    [0, 0, 0, 1, 1, 1, 1, 0]]))

Note which `None` goes where. Swap them and you get the transpose.

That bug is invisible when `Tq == 1`, which is every decode step. It only
shows up in prefill. You will be made to find it in **1l**.

## Next

**1g** — collapsing an axis, and the argument that makes the result line
back up.